<a href="https://colab.research.google.com/github/Lupo6789/tesi_scacchi/blob/main/03_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Obiettivo e Metodologia della Baseline Avanzata

In questa sezione definisco le prestazioni di riferimento (Baseline) utilizzando esclusivamente le informazioni note *prima* della prima mossa. Lo scopo è quantificare il potere predittivo dei metadati per stabilire un benchmark statistico e probabilistico rigoroso.
Per garantire la massima solidità metodologica, l'analisi segue questi passaggi:

**Pipeline di Standardizzazione**: Applico uno StandardScaler integrato in una *Pipeline di scikit-learn*. Questo passaggio normalizza le distribuzioni delle feature numeriche, garantendo una convergenza ottimale della Regressione Logistica ed evitando distorsioni legate alla diversa scala dei metadati.
Confronto tra Scenari (A vs B):
Scenario A: Il modello effettua le previsioni conoscendo unicamente il divario di abilità teorica.
Scenario B (Metadati Completi): Il modello utilizza tutte le informazioni numeriche pre-partita (differenza Elo, rating assoluti, tempo sul timer e validità per il punteggio).
   Entrambi gli scenari vengono valutati tramite Accuratezza, ROC-AUC e Log-Loss per misurare la reale calibrazione delle probabilità.
**Misurazione della Variabilità:** Applico una Cross-Validation stratificata a 5-Fold sul Train Set per estrarre l'accuratezza sotto forma di media e deviazione standard, isolando il rumore statistico del campione.
Analisi Teorica (De-standardizzazione): Estraggo il coefficiente della differenza Elo assegnato dal modello. Tramite un'operazione di de-standardizzazione matematica, riporto il coefficiente alla sua scala originale per poterlo confrontare direttamente con il valore teorico della formula di Arpad Elo (0.00576).

# Obiettivo e Metodologia della Baseline Avanzata

In questa sezione definisco le prestazioni di riferimento (Baseline) utilizzando esclusivamente le informazioni note prima della prima mossa. Lo scopo è quantificare il potere predittivo dei metadati per stabilire un benchmark statistico e probabilistico rigoroso prima di introdurre le feature posizionali.

Per garantire la massima solidità metodologica, l'analisi segue questi passaggi:

**Punto di Riferimento Minimo (Dummy Classifier):** Reinserisco un `DummyClassifier` come livello zero della validazione. Questo algoritmo funge da "pavimento" statistico, prevedendo sempre la classe maggioritaria (il Bianco) e restituendo una probabilità statica (AUC = 0.500), quantificando di fatto il totale affidamento al caso.
 **Pipeline di Standardizzazione:** Applico uno `StandardScaler` integrato in una Pipeline di scikit-learn. Questo passaggio normalizza le distribuzioni delle feature numeriche, garantendo una convergenza ottimale della Regressione Logistica ed evitando distorsioni legate alla diversa scala dei metadati.
 **Confronto tra Scenari (A vs B):**
  * **Scenario A:** Il modello effettua le previsioni conoscendo unicamente il divario di abilità teorica (`rating_diff`).
  * **Scenario B (Metadati Completi):** Il modello utilizza tutte le informazioni numeriche pre-partita (differenza Elo, rating assoluti, tempo sul timer e validità per il punteggio).
* **Misurazione della Variabilità:** Applico una Cross-Validation stratificata a 5-Fold sul Train Set per estrarre l'accuratezza sotto forma di media e deviazione standard, isolando il rumore statistico del campione.
* **Analisi Teorica (De-standardizzazione):** Estraggo il coefficiente della differenza Elo assegnato dal modello. Tramite un'operazione di de-standardizzazione matematica, riporto il coefficiente alla sua scala originale per poterlo confrontare direttamente con il valore teorico della formula di Arpad Elo.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings

# Disabilita i warning di convergenza per mantenere pulito l'output
warnings.filterwarnings('ignore')

print("1. VALUTAZIONE BASELINE AVANZATA E DUMMY CLASSIFIER\n")

soglie = [10, 20, 30, 40]
risultati_cv = []
risultati_test = []

# Imposta la Cross-Validation a 5 fette e le metriche complete
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'acc': 'accuracy', 'roc': 'roc_auc', 'loss': 'neg_log_loss'}

# Inizializza il modello logistico incapsulato in una Pipeline con StandardScaler
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=42, max_iter=1000))
])

# Inizializza il Dummy Classifier (prevede sempre in base alla classe maggioritaria / probabilità a priori)
dummy = DummyClassifier(strategy='prior')

for N in soglie:
    # Carica i dataset filtrati per la soglia N (già dotati di rating_diff e rating_mean dal Notebook 02)
    df_train = pd.read_csv(f'train_features_N{N}.csv')
    df_test = pd.read_csv(f'test_features_N{N}.csv')

    # Definisce il target
    y_train = df_train['winner']
    y_test = df_test['winner']

    # Definisce gli scenari pre-partita (senza white_rating e black_rating, ora sostituiti da rating_diff e rating_mean)
    features_b = ['rating_diff', 'rating_mean', 'rated', 'tempo_stimato_sec']

    X_train_a = df_train[['rating_diff']]
    X_test_a = df_test[['rating_diff']]

    X_train_b = df_train[features_b]
    X_test_b = df_test[features_b]

    # --- SCENARIO ZERO: DUMMY CLASSIFIER ---
    cv_dum = cross_validate(dummy, X_train_a, y_train, cv=cv, scoring=scoring)

    dummy.fit(X_train_a, y_train)
    y_prob_dum = dummy.predict_proba(X_test_a)[:, 1]
    acc_dum = accuracy_score(y_test, dummy.predict(X_test_a))
    roc_dum = roc_auc_score(y_test, y_prob_dum)
    loss_dum = log_loss(y_test, y_prob_dum)

    # --- SCENARIO A: Solo Rating Diff ---
    cv_a = cross_validate(pipeline, X_train_a, y_train, cv=cv, scoring=scoring)

    pipeline.fit(X_train_a, y_train)
    y_pred_a = pipeline.predict(X_test_a)
    y_prob_a = pipeline.predict_proba(X_test_a)[:, 1]

    acc_a = accuracy_score(y_test, y_pred_a)
    roc_a = roc_auc_score(y_test, y_prob_a)
    loss_a = log_loss(y_test, y_prob_a)

    # Estrae i pesi matematici solo al primo ciclo (N=10)
    if N == 10:
        scaler_a = pipeline.named_steps['scaler']
        model_a = pipeline.named_steps['logreg']

        # De-standardizza il coefficiente per renderlo confrontabile con la formula teorica
        coef_elo = model_a.coef_[0][0] / scaler_a.scale_[0]
        intercetta_bianco = model_a.intercept_[0] - (model_a.coef_[0][0] * scaler_a.mean_[0] / scaler_a.scale_[0])

    # --- SCENARIO B: Tutti i Metadati Pre-Partita ---
    cv_b = cross_validate(pipeline, X_train_b, y_train, cv=cv, scoring=scoring)

    pipeline.fit(X_train_b, y_train)
    y_pred_b = pipeline.predict(X_test_b)
    y_prob_b = pipeline.predict_proba(X_test_b)[:, 1]

    acc_b = accuracy_score(y_test, y_pred_b)
    roc_b = roc_auc_score(y_test, y_prob_b)
    loss_b = log_loss(y_test, y_prob_b)

    # Salva la riga riassuntiva in Cross-Validation (riferimento metodologico principale)
    risultati_cv.append({
        'N': N,
        'CV_Acc_Dummy (%)': f"{cv_dum['test_acc'].mean()*100:.2f} ± {cv_dum['test_acc'].std()*100:.2f}",
        'CV_LogLoss_Dummy': round(-cv_dum['test_loss'].mean(), 3),
        'CV_Acc_A (%)': f"{cv_a['test_acc'].mean()*100:.2f} ± {cv_a['test_acc'].std()*100:.2f}",
        'CV_ROC_A': round(cv_a['test_roc'].mean(), 3),
        'CV_LogLoss_A': round(-cv_a['test_loss'].mean(), 3),
        'CV_Acc_B (%)': f"{cv_b['test_acc'].mean()*100:.2f} ± {cv_b['test_acc'].std()*100:.2f}",
        'CV_ROC_B': round(cv_b['test_roc'].mean(), 3),
        'CV_LogLoss_B': round(-cv_b['test_loss'].mean(), 3),
    })

    # Salva la riga riassuntiva sul Test Set (riferimento out-of-sample dichiarato)
    risultati_test.append({
        'N': N,
        'Test_Acc_Dummy (%)': round(acc_dum * 100, 2),
        'Test_LogLoss_Dummy': round(loss_dum, 3),
        'Test_Acc_A (%)': round(acc_a * 100, 2),
        'Test_ROC_AUC_A': round(roc_a, 3),
        'Test_LogLoss_A': round(loss_a, 3),
        'Test_Acc_B (%)': round(acc_b * 100, 2),
        'Test_ROC_AUC_B': round(roc_b, 3),
        'Test_LogLoss_B': round(loss_b, 3),
    })

print("--- A) RISULTATI IN CROSS-VALIDATION 5-FOLD (TRAINING SET - RIFERIMENTO BASELINE) ---")
print(pd.DataFrame(risultati_cv).to_string(index=False))

print("\n--- B) RIFERIMENTO OUT-OF-SAMPLE SUL TEST SET (SOLO SCOPO INFORMATIVO) ---")
print(pd.DataFrame(risultati_test).to_string(index=False))

print("\n2. ANALISI MATEMATICA E CONCLUSIONI (N=10)")
print(f"Coefficiente Teorico Elo (Formula di Arpad Elo: ln(10)/400): 0.00576")
print(f"Coefficiente Calcolato (rating_diff):                        {coef_elo:.5f}")
print(f"Intercetta (Vantaggio Innato del Bianco):                    {intercetta_bianco:.5f}")

1. VALUTAZIONE BASELINE AVANZATA E DUMMY CLASSIFIER

--- A) RISULTATI IN CROSS-VALIDATION 5-FOLD (TRAINING SET - RIFERIMENTO BASELINE) ---
 N CV_Acc_Dummy (%)  CV_LogLoss_Dummy CV_Acc_A (%)  CV_ROC_A  CV_LogLoss_A CV_Acc_B (%)  CV_ROC_B  CV_LogLoss_B
10     52.22 ± 0.01             0.692 64.72 ± 0.40     0.713         0.618 64.71 ± 0.43     0.713         0.619
20     51.92 ± 0.01             0.692 64.27 ± 0.66     0.710         0.621 64.32 ± 0.71     0.709         0.621
30     51.65 ± 0.01             0.693 64.02 ± 0.87     0.705         0.624 63.98 ± 0.70     0.705         0.625
40     51.26 ± 0.00             0.693 63.50 ± 0.90     0.699         0.629 63.57 ± 0.93     0.699         0.629

--- B) RIFERIMENTO OUT-OF-SAMPLE SUL TEST SET (SOLO SCOPO INFORMATIVO) ---
 N  Test_Acc_Dummy (%)  Test_LogLoss_Dummy  Test_Acc_A (%)  Test_ROC_AUC_A  Test_LogLoss_A  Test_Acc_B (%)  Test_ROC_AUC_B  Test_LogLoss_B
10               52.09               0.692           65.15           0.714           0

### Osservazione sul Campione Statistico al variare di N
Analizzando i risultati per gli Scenari A e B, si osserva una lieve ma costante flessione delle metriche predittive (Accuracy e ROC-AUC) al crescere della soglia N.
È importante sottolineare che all'aumentare di N, il campione statistico analizzato subisce un cambiamento strutturale. I dataset generati per soglie più elevate, infatti, contengono esclusivamente le partite la cui durata supera le N semimosse. Si ipotizza che, di conseguenza, il modello valutato a N=40 non stia analizzando la stessa popolazione di partite vista a N=10, ma un sottoinsieme specifico: incontri mediamente più lunghi e strategicamente equilibrati, in cui i giocatori hanno superato l'apertura senza commettere errori fatali immediati. In un campione scremato e bilanciato in questo modo, è del tutto naturale che il solo divario teorico iniziale (Elo) perda una frazione del suo potere predittivo.

### Osservazioni Metodologiche e Matematiche

- **Modello A vs B:** L'aggiunta di tutti i metadati pre-partita (Scenario B) non produce miglioramenti rispetto al solo `rating_diff` (Scenario A). Questo è un risultato in sé: la differenza di Elo assorbe la quasi totalità del potere predittivo pre-partita, confermando lo Scenario A come la Baseline d'elezione per i successivi confronti.
- **Derivazione della Costante Teorica:** Il valore teorico di `0.00576` non è casuale, ma deriva dalla formula logistica originale ideata da Arpad Elo per calcolare il valore atteso. Matematicamente, tale coefficiente è dato dal calcolo $\ln(10) / 400 \approx 0.005756$.
- **Ipotesi sul Coefficiente Elo Empirico:** Il coefficiente calcolato empiricamente dal nostro modello (`0.00382`) risulta sensibilmente inferiore a quello teorico (`0.00576`), con uno scarto di circa il **34% in meno**. Come ipotesi esplicativa, questa discrepanza potrebbe dipendere dal rumore di fondo e dalla varianza intrinseca nei rating online (dinamica del sistema Glicko-2, account nuovi o in fase di calibrazione rapida): la presenza di rumore di misura nei rating tende infatti ad **attenuare il coefficiente stimato**, appiattendo la curva delle probabilità osservate rispetto all'aspettativa teorica ideale.
- **Interpretazione dell'Intercetta (Il Vantaggio del Bianco):** La destandardizzazione ha isolato un'intercetta di base pari a circa $0.075$. Inviando questo valore alla funzione sigmoide della Regressione Logistica, ricaviamo la probabilità di base $P = \frac{1}{1 + e^{-0.075}} \approx 0.5187$. Questo dato quantifica scientificamente il vantaggio del tratto: a perfetta parità di Elo ($x = 0$), il giocatore con il Bianco vanta matematicamente una probabilità di vittoria del **51.9%**.

### Osservazione sul Campione Statistico al variare di $N$

Analizzando i risultati in Cross-Validation per gli Scenari A e B, si osserva una lieve ma costante flessione delle metriche predittive al crescere della soglia $N$: nello Scenario A, l'Accuracy media scende dal **64.72%** ($N = 10$) al **63.50%** ($N = 40$), la ROC-AUC passa da **0.713** a **0.699** e la Log-Loss sale da **0.618** a **0.629**.

Questo andamento riflette un cambiamento strutturale del campione statistico. Per ciascuna soglia $N$, il dataset include infatti solo le partite di durata strettamente superiore a $N$ semimosse, ancora in corso a quel punto. Di conseguenza, a $N = 40$ il modello non valuta la stessa popolazione vista a $N = 10$, ma un sottoinsieme progressivamente scremato verso incontri più lunghi e combattuti, in cui entrambi i giocatori hanno superato l'apertura senza commettere errori fatali immediati. In una popolazione più equilibrata è naturale che il solo divario teorico iniziale (Elo) perda una frazione del proprio potere predittivo. Lo stesso effetto emerge anche nel `DummyClassifier`, la cui Accuracy in CV cala dal **52.22%** ($N = 10$) al **51.26%** ($N = 40$), segnalando una leggera attenuazione del vantaggio strutturale del Bianco nelle partite più lunghe.

### Osservazioni Metodologiche e Matematiche

- **Modello A vs B:** l'aggiunta degli altri metadati pre-partita (`rating_mean`, `rated`, `tempo_stimato_sec`) nello Scenario B non produce alcun miglioramento rispetto al solo `rating_diff` (Scenario A). Su tutte le soglie, Accuracy, ROC-AUC e Log-Loss in Cross-Validation risultano sostanzialmente sovrapponibili (con scarti nell'ordine dei centesimi di punto percentuale, ben all'interno della deviazione standard dei fold). In un modello lineare privo di interazioni, queste variabili simmetriche rispetto ai due colori non aggiungono potere discriminante: la differenza di Elo assorbe la quasi totalità del segnale pre-partita, confermando lo Scenario A come *Baseline* di riferimento.
- **Coerenza tra Cross-Validation e Test Set:** le metriche *out-of-sample* sul Test Set (riportate a scopo puramente informativo e comparativo) ricalcano fedelmente le stime ottenute in Cross-Validation su tutte le soglie (ad esempio, a $N = 10$ nello Scenario A: Accuracy **65.15%** vs **64.72%**, ROC-AUC **0.714** vs **0.713**, Log-Loss **0.616** vs **0.618**), confermando l'assenza di *overfitting* e la stabilità della *Baseline*.
- **Derivazione della Costante Teorica:** il coefficiente teorico di riferimento (`0.00576`) deriva direttamente dalla formulazione logistica originale di Arpad Elo per il punteggio atteso, ed è pari a $\ln(10) / 400 \approx 0.005756$.
- **Ipotesi sul Coefficiente Elo Empirico:** il coefficiente stimato empiricamente dal modello (`0.00382`) risulta sensibilmente inferiore a quello teorico (`0.00576`), con una riduzione di circa il **34%**. Una spiegazione plausibile risiede nell'attenuazione statistica indotta dal rumore di misura nei rating online (dinamiche del sistema Glicko-2 su Lichess, account provvisori o in calibrazione rapida, presenza di partite non classificate e maggiore variabilità tattica nelle cadenze *Rapid*): tali fattori tendono ad appiattire la curva delle probabilità osservate rispetto alla relazione teorica ideale.
- **Interpretazione dell'Intercetta (Vantaggio del Bianco):** la de-standardizzazione isola un'intercetta pari a **0.07523**. Applicando la funzione sigmoide della Regressione Logistica a parità di rating (`rating_diff = 0`), si ottiene $P = \frac{1}{1 + e^{-0.07523}} \approx 0.5188$: a parità di forza teorica, il giocatore con il Bianco dispone quindi di una probabilità di vittoria stimata pari al **51.9%**, che quantifica empiricamente il vantaggio del primo tratto.